# MeetingActionAgent：双智能体会议纪要助手

> 输入会议文字记录，由 MinutesAgent 提取纪要，再由 ReviewAgent 对照原文审核。

作者：[@Henry2513](https://github.com/Henry2513)

日期：2026-08-04

**适合读者**
- 第一次学习 Agent 或 HelloAgents 的开发者
- 希望理解“生成 Agent + 审核 Agent”协作方式的学习者

**前置条件**
- Python 3.11+
- 已安装 `requirements.txt`
- 真实运行时需要一个 OpenAI-compatible LLM API

**学习目标**
- 使用两个 `SimpleAgent` 顺序协作
- 用 Pydantic 校验模型返回的 JSON
- 限制重试次数并生成 JSON、Markdown 两种结果


## 学习路线

1. 加载环境与项目路径
2. 定义会议纪要数据结构
3. 解析并渲染结构化结果
4. 创建 MinutesAgent 和 ReviewAgent
5. 编排提取、审核和一次修正
6. 运行真实双 Agent 流程和自检


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, SimpleAgent
from pydantic import BaseModel, Field, ValidationError

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
print(f"项目目录: {PROJECT_ROOT}")


## 1. 双 Agent 架构

```text
会议文字记录
  → MinutesAgent：只提取原文支持的信息
  → ReviewAgent：检查遗漏、编造和冲突
  → 必要时修正并复核一次
  → JSON + Markdown
```

第一版不使用工具调用。普通 Python 代码负责读取、校验和保存文件，Agent 只负责语言理解与审核。


In [ ]:
# 表示从会议原文中提取的一条行动项。
class ActionItem(BaseModel):
    task: str = Field(min_length=1)
    owner: str | None = None
    due_date_raw: str | None = None
    priority: Literal["高", "中", "低", "未说明"] = "未说明"
    evidence: str = Field(min_length=1)


# 表示最终输出的完整会议纪要及其审核状态。
class MeetingResult(BaseModel):
    title: str = Field(min_length=1)
    meeting_date: str | None = None
    participants: list[str] = Field(default_factory=list)
    summary: str = Field(min_length=1)
    decisions: list[str] = Field(default_factory=list)
    action_items: list[ActionItem] = Field(default_factory=list)
    open_questions: list[str] = Field(default_factory=list)
    review_status: Literal["pending", "passed", "needs_manual_review"] = "pending"
    review_issues: list[str] = Field(default_factory=list)


# 表示 ReviewAgent 对纪要草稿的审核结论和问题。
class ReviewResult(BaseModel):
    passed: bool
    issues: list[str] = Field(default_factory=list)
    missing_items: list[str] = Field(default_factory=list)
    unsupported_items: list[str] = Field(default_factory=list)
    revision_advice: list[str] = Field(default_factory=list)



## 2. 解析 Agent 返回的 JSON

模型有时会把 JSON 包在 Markdown 代码围栏中，或者在前后添加一句解释。下面的函数先提取最外层 JSON 对象，再交给 Pydantic 校验。


In [ ]:
# 从模型响应中截取并解析 JSON 对象。
def extract_json_object(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("模型响应中没有完整的 JSON 对象")
    return json.loads(text[start : end + 1])


# 将模型响应解析并验证为指定的 Pydantic 模型。
def parse_model_response(text: str, model_type: type[BaseModel]) -> BaseModel:
    return model_type.model_validate(extract_json_object(text))


## 3. 把结构化结果转换为 Markdown

Markdown 由普通 Python 生成，避免让模型重复改写已经审核过的内容。


In [ ]:
# 将可空文本整理成适合 Markdown 表格的内容。
def markdown_cell(value: str | None) -> str:
    if not value:
        return "未提供"
    return value.replace("|", "\\|").replace("\n", " ")


# 将会议纪要转换成 Markdown 文本。
def to_markdown(result: MeetingResult) -> str:
    meeting_date = markdown_cell(result.meeting_date)
    participants = "、".join(result.participants) if result.participants else "未提供"
    lines = [
        f"# {result.title}",
        "",
        f"**会议日期：** {meeting_date}",
        "",
        f"**参会者：** {participants}",
        "",
        "## 会议摘要",
        "",
        result.summary,
        "",
        "## 已确认决策",
        "",
    ]
    lines.extend([f"- {item}" for item in result.decisions] or ["- 无"])
    lines.extend([
        "",
        "## 行动项",
        "",
        "| 任务 | 负责人 | 截止日期（原文） | 优先级 | 原文证据 |",
        "|---|---|---|---|---|",
    ])
    if result.action_items:
        for item in result.action_items:
            lines.append(
                "| "
                + " | ".join([
                    markdown_cell(item.task),
                    markdown_cell(item.owner),
                    markdown_cell(item.due_date_raw),
                    markdown_cell(item.priority),
                    markdown_cell(item.evidence),
                ])
                + " |"
            )
    else:
        lines.append("| 无 | 未提供 | 未提供 | 未说明 | 未提供 |")

    lines.extend(["", "## 待确认问题", ""])
    lines.extend([f"- {item}" for item in result.open_questions] or ["- 无"])
    lines.extend(["", "## 审核状态", "", f"`{result.review_status}`"])
    if result.review_issues:
        lines.extend(["", "### 审核问题", ""])
        lines.extend([f"- {item}" for item in result.review_issues])
    return "\n".join(lines) + "\n"


# 将会议结果保存为 JSON 和 Markdown 文件。
def save_result(result: MeetingResult, stem: str = "meeting_result") -> tuple[Path, Path]:
    output_dir = PROJECT_ROOT / "outputs"
    json_path = output_dir / f"{stem}.json"
    markdown_path = output_dir / f"{stem}.md"
    json_path.write_text(result.model_dump_json(indent=2), encoding="utf-8")
    markdown_path.write_text(to_markdown(result), encoding="utf-8")
    return json_path, markdown_path


## 4. Agent 职责与提示词

- MinutesAgent 只能提取原文支持的信息，未知字段必须保持为空。
- ReviewAgent 必须同时查看原文和草稿，重点检查遗漏、编造、日期冲突，以及“建议”是否被误写为“决定”。


In [ ]:
MINUTES_SYSTEM_PROMPT = """你是严谨的中文会议纪要提取专家。
只使用会议原文明确支持的信息，不得补充常识或猜测。
严格区分讨论、建议和已确认决策。
每个行动项必须保留一段原文证据；未知负责人、会议日期或截止日期必须为 null。
只返回符合用户给定 Schema 的 JSON，不要返回 Markdown 或额外解释。"""

REVIEW_SYSTEM_PROMPT = """你是独立的会议纪要审核员。
必须逐项对照会议原文和纪要草稿，检查遗漏、编造、模糊行动项和日期冲突。
不能因为文字通顺就判定通过，也不能使用外部信息。
只返回符合用户给定 Schema 的 JSON，不要返回 Markdown 或额外解释。"""


## 5. 有限调用与格式修复

整个流程共享四次模型调用预算。JSON 首次解析失败时允许请求一次格式修复，但修复同样计入预算。


In [ ]:
class CallBudget:
    # 初始化模型调用次数上限。
    def __init__(self, maximum: int = 4) -> None:
        self.maximum = maximum
        self.used = 0

    # 计算剩余的模型调用次数。
    @property
    def remaining(self) -> int:
        return self.maximum - self.used

    # 在次数限制内执行一次 Agent 调用。
    def run(self, agent, prompt: str) -> str:
        if self.remaining <= 0:
            raise RuntimeError("已达到四次模型调用上限")
        self.used += 1
        return agent.run(prompt)


# 调用 Agent 并将响应解析为指定的数据模型。
def run_structured(
    agent,
    prompt: str,
    model_type: type[BaseModel],
    budget: CallBudget,
) -> BaseModel:
    schema = json.dumps(model_type.model_json_schema(), ensure_ascii=False)
    full_prompt = f"{prompt}\n\n必须遵循以下 JSON Schema：\n{schema}"
    raw_response = budget.run(agent, full_prompt)
    try:
        return parse_model_response(raw_response, model_type)
    except ValueError as error:
        if budget.remaining <= 0:
            raise RuntimeError(f"JSON 校验失败且没有剩余调用次数：{error}") from error
        repair_prompt = (
            "上一次响应无法通过 JSON 校验。不要改变内容含义，只修复格式。\n"
            f"校验错误：{error}\n"
            f"原响应：\n{raw_response}\n"
            f"目标 Schema：\n{schema}\n"
            "只返回修复后的 JSON。"
        )
        repaired_response = budget.run(agent, repair_prompt)
        return parse_model_response(repaired_response, model_type)


# 创建 MinutesAgent 和 ReviewAgent。
def build_agents():
    llm = HelloAgentsLLM()
    minutes_agent = SimpleAgent(name="MinutesAgent", llm=llm, system_prompt=MINUTES_SYSTEM_PROMPT)
    review_agent = SimpleAgent(name="ReviewAgent", llm=llm, system_prompt=REVIEW_SYSTEM_PROMPT)
    return minutes_agent, review_agent


## 6. 完整分析流程

首次审核通过时只调用两次模型；未通过且仍有两次预算时，MinutesAgent 修正一次，再由 ReviewAgent 最终复核。


In [ ]:
# 检查并清理输入的会议文本。
def validate_transcript(transcript: str) -> str:
    cleaned = transcript.strip()
    if len(cleaned) < 20:
        raise ValueError("会议记录过短，请至少提供 20 个字符")
    return cleaned


# 组合审核会议纪要所需的提示词。
def make_review_prompt(transcript: str, draft: MeetingResult) -> str:
    return (
        "请审核以下会议纪要草稿。\n\n"
        f"【会议原文】\n{transcript}\n\n"
        f"【纪要草稿】\n{draft.model_dump_json(indent=2)}"
    )


# 执行纪要提取、审核和必要时修正的完整流程。
def analyze_meeting(transcript: str) -> tuple[MeetingResult, ReviewResult, int]:
    transcript = validate_transcript(transcript)
    minutes_agent, review_agent = build_agents()
    budget = CallBudget(maximum=4)

    draft_prompt = f"请从以下会议原文提取结构化纪要：\n\n{transcript}"
    draft = run_structured(minutes_agent, draft_prompt, MeetingResult, budget)
    review = run_structured(review_agent, make_review_prompt(transcript, draft), ReviewResult, budget)

    if review.passed:
        final_result = draft.model_copy(update={"review_status": "passed", "review_issues": []})
        return final_result, review, budget.used

    issues = review.issues + review.missing_items + review.unsupported_items
    if budget.remaining < 2:
        final_result = draft.model_copy(
            update={"review_status": "needs_manual_review", "review_issues": issues}
        )
        return final_result, review, budget.used

    revision_prompt = (
        "请根据审核意见修正纪要。仍然只能使用会议原文支持的信息。\n\n"
        f"【会议原文】\n{transcript}\n\n"
        f"【原草稿】\n{draft.model_dump_json(indent=2)}\n\n"
        f"【审核意见】\n{review.model_dump_json(indent=2)}"
    )
    revised = run_structured(minutes_agent, revision_prompt, MeetingResult, budget)
    final_review = run_structured(review_agent, make_review_prompt(transcript, revised), ReviewResult, budget)
    final_issues = final_review.issues + final_review.missing_items + final_review.unsupported_items
    status = "passed" if final_review.passed else "needs_manual_review"
    final_result = revised.model_copy(update={"review_status": status, "review_issues": final_issues})
    return final_result, final_review, budget.used


## 7. 运行双 Agent 会议示例

本单元格直接读取模型配置并执行 MinutesAgent、ReviewAgent。运行前必须在 `.env` 中填写 `LLM_MODEL_ID`、`LLM_API_KEY` 和 `LLM_BASE_URL`。


In [ ]:
sample_transcript = (PROJECT_ROOT / "data" / "sample_meeting.txt").read_text(encoding="utf-8")
result, _, call_count = analyze_meeting(sample_transcript)
json_path, markdown_path = save_result(result)
print(f"模型调用次数: {call_count}")
print(f"审核状态: {result.review_status}")
print(f"已保存: {json_path.name}, {markdown_path.name}")


## 8. 结构与编排自检

这些检查不调用模型，验证 Pydantic 结构、JSON 提取、缺失字段、空行动项、空输入和 Markdown 渲染。


In [ ]:
expected_path = PROJECT_ROOT / "outputs" / "example_result.json"
expected_result = MeetingResult.model_validate_json(expected_path.read_text(encoding="utf-8"))

fenced = "```json\n" + expected_result.model_dump_json() + "\n```"
assert parse_model_response(fenced, MeetingResult).title == "新用户注册功能迭代"
assert expected_result.meeting_date == "2026-07-27"
assert expected_result.action_items[-1].owner is None

empty_actions = MeetingResult(
    title="信息同步会",
    participants=[],
    summary="本次会议仅同步信息，没有形成行动项。",
    decisions=[],
    action_items=[],
    open_questions=[],
)
assert empty_actions.action_items == []
assert "| 无 |" in to_markdown(empty_actions)

try:
    validate_transcript("太短")
except ValueError:
    pass
else:
    raise AssertionError("过短会议记录应被拒绝")

try:
    ActionItem(task="测试", owner=None, due_date_raw=None, priority="紧急", evidence="原文")
except ValidationError:
    pass
else:
    raise AssertionError("非法优先级应被 Pydantic 拒绝")

rendered = to_markdown(expected_result)
tracked_markdown = (PROJECT_ROOT / "outputs" / "example_minutes.md").read_text(encoding="utf-8")
assert rendered == tracked_markdown
print("自检通过：6 组")


## 练习：分析边界会议

打开 `data/edge_case_meeting.txt`，先人工预测结果：

1. “可以考虑下个月上线”是否属于已确认决策？
2. 测试环境确认任务是否有负责人？
3. 产品文档的日期是否存在冲突？

启用真实模型后，可以把下面的 `edge_transcript` 传给 `analyze_meeting`，再与预测比较。


In [ ]:
edge_transcript = (PROJECT_ROOT / "data" / "edge_case_meeting.txt").read_text(encoding="utf-8")
answer_scaffold = {
    "confirmed_launch_decision": False,
    "test_environment_owner": None,
    "document_date_conflict": True,
}
answer_scaffold


## 常见问题与下一步

- **把建议写成决定**：Reviewer 必须检查“考虑、建议、可能”等措辞。
- **编造负责人或日期**：未知值保持 `null`，最终 Markdown 显示“未提供”。
- **JSON 不稳定**：允许一次格式修复，但仍受四次调用预算限制。
- **多份会议连续分析**：每次调用 `analyze_meeting` 都会创建新的 Agent，避免历史记录互相污染。

第二版可增加日期标准化工具，但第一版保持双 Agent、无工具调用。
